<a href="https://colab.research.google.com/github/org-cyber/Avo/blob/master/vlada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl", split="train")
print(f"✅ Loaded {len(dataset)} training examples")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Loaded 250 training examples


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    use_gradient_checkpointing = "unsloth",
)

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
def format_example(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True   # adds the prompt for the assistant reply
    )
    return {"text": text}

dataset = dataset.map(format_example)

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    output_dir = "outputs",
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",       # use the pre‑formatted column
    max_seq_length = 2048,
    args = training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.187769
2,3.148129
3,3.190606
4,2.951048
5,2.660383
6,2.604003
7,2.106098
8,1.861561
9,1.714863
10,1.482772


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.8506526639064153, metrics={'train_runtime': 91.75, 'train_samples_per_second': 5.232, 'train_steps_per_second': 0.654, 'total_flos': 656674355933184.0, 'train_loss': 0.8506526639064153, 'epoch': 1.896})

In [ ]:
def test(prompt):
    messages = [
        {"role": "system", "content": "You are Vlada, an expert API security analyst. Analyze the request fingerprint and classify it as BENIGN or MALICIOUS. Provide a concise, technical reason based on the given heuristic flags and request metadata."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test 1 — Obvious XSS attempt
print("Test 1 (XSS):")
print(test("Method: GET, Path: /search, Query Keys: [query], Body Keys: [], Heuristic Flags: Reflected XSS pattern in 'query' parameter, Risk Score: 84, User-Agent: curl/7.68.0, Client IP: 10.0.0.1"))

# Test 2 — Benign request
print("\nTest 2 (Benign):")
print(test("Method: GET, Path: /home, Query Keys: [], Body Keys: [], Heuristic Flags: None, Risk Score: 0, User-Agent: Mozilla/5.0, Client IP: 192.168.1.1"))

# Test 3 — Path traversal
print("\nTest 3 (Path Traversal):")
print(test("Method: GET, Path: /download, Query Keys: [file], Body Keys: [], Heuristic Flags: Path traversal pattern in 'file' parameter, Risk Score: 85, User-Agent: python-requests/2.31.0, Client IP: 172.16.0.5"))

# Test 4 — Command Injection
print("\nTest 4 (Command Injection):")
print(test("Method: POST, Path: /api/exec, Query Keys: [], Body Keys: [cmd], Heuristic Flags: Command injection in 'cmd' body field, Risk Score: 93, User-Agent: zgrab/0.x, Client IP: 10.10.10.10"))

# Test 5 — A tricky case: low risk score but suspicious flag
print("\nTest 5 (Borderline):")
print(test("Method: GET, Path: /api/search, Query Keys: [q], Body Keys: [], Heuristic Flags: None, Risk Score: 5, User-Agent: PostmanRuntime/7.32.0, Client IP: 203.0.113.5"))

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test 1 (XSS):


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

MALICIOUS. Reason: The 'query' parameter appears to be user-input that could be used for DOM-based XSS via JavaScript URL execution. This is a classic web attack vector.

Test 2 (Benign):


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BENIGN. Reason: Normal API request with no malicious heuristics triggered.

Test 3 (Path Traversal):


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MALICIOUS. Reason: The 'file' parameter appears to be attempting a path traversal (../../../..). This could lead to reading arbitrary files from the server.

Test 4 (Command Injection):


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MALICIOUS. Reason: The 'cmd' parameter is being used to execute arbitrary commands on the server. This endpoint should be strictly protected against command injection attacks.

Test 5 (Borderline):
BENIGN. Reason: Normal API request with no malicious heuristics triggered.


In [ ]:
# Merge LoRA weights into the base model
merged = model.merge_and_unload()

# Login to Hugging Face (you'll need a token with write access)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6730f94eb4215fd388569fbb', 'name': 'pixelpro2008', 'fullname': 'Pixel Pro', 'email': 'chinwokennedy9@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/Fw8ldGWUdm3UurzV64Ky4.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab-upload', 'role': 'write', 'createdAt': '2026-04-25T12:38:15.331Z'}}}


In [ ]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6730f94eb4215fd388569fbb', 'name': 'pixelpro2008', 'fullname': 'Pixel Pro', 'email': 'chinwokennedy9@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/Fw8ldGWUdm3UurzV64Ky4.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab-upload', 'role': 'write', 'createdAt': '2026-04-25T12:38:15.331Z'}}}


In [ ]:
model.push_to_hub("pixelpro2008/vlada-v2-lora")
print("✅ Adapter pushed successfully")

README.md:   0%|          | 0.00/586 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########|  40.0B /  40.0B            

Saved model to https://huggingface.co/pixelpro2008/vlada-v2-lora
✅ Adapter pushed successfully
